In [1]:
import sys
import logging
from pathlib import Path

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline_handler import PipelineHandler

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# Percorsi
sidecar_path = project_root / "data/processed/armstrong/sc_multi_pret_distanceclasses1.json"
queries_path = project_root / "data/queries/armstrong/eval_queries.json"

# Inizializzazione gestore pipeline
handler = PipelineHandler(
    sidecar_path=sidecar_path,
    queries_path=queries_path
)

print("!>> PipelineHandler e SidecarManager inizializzati con successo.")

/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(
13:56:43 [INFO] httpx: HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
13:56:43 [INFO] src.pipeline_handler: ?> Pipeline Inizializzata


!>> PipelineHandler e SidecarManager inizializzati con successo.


In [14]:
### 1 ### Impostazione Parametri
handler.set_parameters(
    top_j=5,
    top_k=7,
    top_n=5,
    order_by=["DIST", "TAGS"]
)

14:05:38 [INFO] src.pipeline_handler: ?> Parametri aggiornati: top_j=5, top_k=7, top_n=5, order_by=['DIST', 'TAGS']


In [15]:
### 2 ### Selezione Query

#Q_AMBIG_01, Q_AMBIG_02, Q_AMBIG_03, Q_SPEC_JAZZ_01, Q_SPEC_BIKE_01, Q_SPEC_ASTRO_01
handler.update_query(new_id="Q_AMBIG_01")

#handler.update_query(new_text="Quali sono i contributi musicali di Louis Armstrong e che strumento suonava?")

14:05:46 [INFO] src.pipeline_handler: ?> Testo Query aggiornato: /home/jovyan/tesi_graphrag/data/queries/armstrong/eval_queries.json--ID>Q_AMBIG_01 
14:05:46 [INFO] src.pipeline_handler: ?> Vettore embedding Query aggiornato: text:Chi è Armstrong e quali sono i principali traguardi raggiunti nella sua carriera?


In [16]:
### 3 ### Tag query
conv_tag_list = ["Astronautica"]
ass_tag_list = [""]
auto_ass_flag = False

handler.update_query_tags(
    new_conv_list=conv_tag_list,
    new_ass_list=ass_tag_list,
    auto_ass_flag=auto_ass_flag,
    print_debug=True
)

14:05:50 [INFO] src.pipeline_handler: ?> Query vettorizzata. Tag assegnati: ['Astronautica']


!>> Tag assegnati alla query:
   >>> Tag di Conversazione: ['Astronautica']
   >>> Tag Manuali della Query: []
   >>> Assegnazione Tag Automatici disabilitata
<<< Lista completa: ['Astronautica']


In [17]:
### 4 ### Retrieval Vettoriale
handler.run_vectorial_retrieval()

14:05:53 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/multilanguage_armstrong/points/query "HTTP/1.1 200 OK"
14:05:53 [INFO] src.pipeline_handler: ?> Recuperati 7 record con relativi vettori di embedding


In [18]:
### 5 ### Reranking
handler.run_top_j_distance_tag_reranking(print_debug=True)

14:05:55 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/multilanguage_armstrong/points "HTTP/1.1 200 OK"


?> [DEBUG INFO] all_reranked:
         >> Il Reranker ha restituito 10 record
    >>> cid: ['neil_armstrong_1']
        |> category: [1]
           |> final_score: [0.615]
    >>> cid: ['neil_armstrong_7']
        |> category: [1]
           |> final_score: [0.5872]
    >>> cid: ['neil_armstrong_2']
        |> category: [2]
           |> final_score: [0.8131]
    >>> cid: ['neil_armstrong_3']
        |> category: [2]
           |> final_score: [0.7134]
    >>> cid: ['louis_armstrong_4']
        |> category: [3]
           |> final_score: [0.5447]
    >>> cid: ['lance_armstrong_2']
        |> category: [4]
           |> final_score: [0.6698]
    >>> cid: ['louis_armstrong_6']
        |> category: [4]
           |> final_score: [0.6627]
    >>> cid: ['lance_armstrong_1']
        |> category: [4]
           |> final_score: [0.6303]
    >>> cid: ['lance_armstrong_3']
        |> category: [4]
           |> final_score: [0.5996]
    >>> cid: ['louis_armstrong_1']
        |> category: [4]
   

In [19]:
### VIEW ### Interfaccia Grafica Interattiva
import ipywidgets as widgets
from IPython.display import display

# Mappatura tra i nomi visibili nei pulsanti e le chiavi interne delle fasi
PHASE_MAP = {
    "1° Retrieval (Vettoriale)": "vectorial",
    "2° Retrieval (Distanze)": "distance",
    "Top-J Reranked": "reranked",
    "All Reranked": "all_reranked"
}

# Selettore orizzontale a schede (ToggleButtons)
phase_selector = widgets.ToggleButtons(
    options=list(PHASE_MAP.keys()),
    description='Fase Grafo:',
    disabled=False,
    button_style='info',
    tooltips=[
        'Mostra i Top-K nodi del retrieval vettoriale iniziale',
        'Mostra i nodi iniziali uniti ai vicini recuperati per distanza',
        'Mostra i Top-J nodi selezionati dal Reranker per l\'LLM',
        'Mostra l\'intero elenco ordinato dal Reranker'
    ],
)

# Container per il rendering dinamico dell'unico widget
output_area = widgets.Output()

def on_phase_change(change):
    selected_label = change['new']
    phase_key = PHASE_MAP[selected_label]
    
    with output_area:
        output_area.clear_output(wait=True)
        # Aggiorna i dati nel widget senza ricreare l'elemento nel DOM
        handler.update_widget_view(phase_key)
        display(handler.widget)

# Registrazione dell'evento sul cambio di selezione
phase_selector.observe(on_phase_change, names='value')

# Inizializzazione della vista sulla prima fase
handler.update_widget_view("vectorial")

# Visualizzazione dell'interfaccia
display(phase_selector)
with output_area:
    display(handler.widget)

14:06:19 [INFO] src.pipeline_handler: ?> Widget aggiornato alla fase 'vectorial' (7 nodi)


ToggleButtons(button_style='info', description='Fase Grafo:', options=('1° Retrieval (Vettoriale)', '2° Retrie…